# Exercises XP: LoRA Implementation Lab
Replace each `TODO` before running the next section.

## What you'll learn

- The fundamentals of LoRA (Low-Rank Adaptation) and why it helps churn out efficient fine-tunes.
- How to implement LoRA matrices `A` and `B`, plus how to wrap existing `nn.Linear` layers.
- Differences between standard linear layers, LoRA-enhanced layers, and merged-weight alternatives.
- How to freeze base parameters so that only the LoRA adapters receive updates.

## What you will create

- A reusable `LoRALayer` module and two linear wrappers (`LinearWithLoRA`, `LinearWithLoRAMerged`).
- A 3-layer MLP that can be swapped between standard and LoRA-enhanced variants.
- A minimal MNIST training loop plus accuracy helpers to compare frozen vs. fully-trainable adapters.
- A workflow to freeze baseline weights and fine-tune only the LoRA layers.

> **Learning point**  
> Keep the student and teacher notebooks open side by side. Follow the numbered exercises, run setup only once, and watch tensor shapes as you add LoRA adapters.

# Part 0: Environment Setup

Install the CPU-friendly PyTorch stack plus torchvision for MNIST. Reuse caches across reruns to save time.

In [1]:
%pip install --quiet torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

In [2]:
import copy
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

BASE_SEED = 123
torch.manual_seed(BASE_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cpu


# Exercise 1: Implement `LoRALayer`

Create the low-rank matrices `A` and `B`, scale them with `alpha`, and test the module on a toy tensor.

In [3]:
class LoRALayer(nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        # Initialisation de A avec une distribution normale
        std_dev = 1 / torch.sqrt(torch.tensor(rank).float())
        self.A = nn.Parameter(torch.randn(in_dim, rank) * std_dev)
        # Initialisation de B à zéro pour que l'ajustement initial soit nul
        self.B = nn.Parameter(torch.zeros(rank, out_dim))
        self.alpha = alpha

    def forward(self, x):
        # Calcul : x @ A @ B donne le delta. On multiplie par alpha pour le scaling.
        # Dimension : (batch, in_dim) @ (in_dim, rank) @ (rank, out_dim) -> (batch, out_dim)
        x = (x @ self.A @ self.B) * self.alpha
        return x

# Paramètres de test
random_seed = 123
in_dim = 10
out_dim = 5
rank = 2
alpha = 1.0

torch.manual_seed(random_seed)
layer = LoRALayer(in_dim, out_dim, rank, alpha)
x = torch.randn(1, in_dim)

print("Input x:", x)
print("Layer:", layer)
print("LoRA output (should be zeros initially):", layer(x))

Input x: tensor([[-0.4709, -0.4269, -0.0283,  1.4220, -0.3886, -0.8903, -0.9601, -0.4087,
          1.0764, -0.4015]])
Layer: LoRALayer()
LoRA output (should be zeros initially): tensor([[0., 0., 0., 0., 0.]], grad_fn=<MulBackward0>)


# Exercise 2: Wrap `nn.Linear` with LoRA

Combine a frozen linear projection plus a trainable `LoRALayer`. Confirm the adapter outputs add on top of the base logits.

In [4]:
class LinearWithLoRA(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features,
            linear.out_features,
            rank,
            alpha,
        )

    def forward(self, x):
        # L'idée de LoRA : Sortie = Poids_Fixes(x) + Adaptation_Entraînable(x)
        return self.linear(x) + self.lora(x)

base_linear = nn.Linear(in_dim, out_dim)
layer_lora_1 = LinearWithLoRA(base_linear, rank=rank, alpha=alpha)
print("LinearWithLoRA output:", layer_lora_1(x))

LinearWithLoRA output: tensor([[-0.3830,  0.7344,  0.2912, -0.3389,  0.0132]], grad_fn=<AddBackward0>)


# Exercise 3: Swap a simple network layer with LoRA

Start from a single-layer perceptron, then replace its linear block with `LinearWithLoRA`. The outputs should match before training because the LoRA adapters start at zero.

In [5]:
class SingleLayerNet(nn.Module):
    def __init__(self, num_features, num_classes):
        super().__init__()
        self.layer = nn.Linear(num_features, num_classes)

    def forward(self, x):
        return self.layer(x)

single_net = SingleLayerNet(num_features=10, num_classes=5)
sample_input = torch.randn(1, 10)

with torch.no_grad():
    baseline_output = single_net(sample_input)

# Remplacement de la couche linéaire par sa version LoRA
single_net.layer = LinearWithLoRA(single_net.layer, rank=2, alpha=1.0)

with torch.no_grad():
    lora_output = single_net(sample_input)

# Vérification : Comme B est initialisé à 0, lora_output doit être identique à baseline_output
matches = torch.allclose(baseline_output, lora_output)
print("Outputs match before training?", matches)

Outputs match before training? True


# Exercise 4: Merged-weight LoRA layer

Fuse the LoRA matrices with the frozen weights to create a drop-in linear layer that behaves exactly like `LinearWithLoRA`.

In [6]:
class LinearWithLoRAMerged(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features,
            linear.out_features,
            rank,
            alpha,
        )

    def forward(self, x):
        # Fusion mathématique : W_total = W_original + alpha * (A @ B).T
        # On utilise .T car nn.Linear stocke les poids en (out_features, in_features)
        lora_weights = (self.lora.A @ self.lora.B) * self.lora.alpha
        combined_weight = self.linear.weight + lora_weights.T
        return F.linear(x, combined_weight, self.linear.bias)

layer_lora_2 = LinearWithLoRAMerged(nn.Linear(in_dim, out_dim), rank=rank, alpha=alpha)
print("Merged LoRA output:", layer_lora_2(x))

Merged LoRA output: tensor([[-0.6961,  0.4832, -0.1033, -0.0010, -0.2235]],
       grad_fn=<AddmmBackward0>)


# Exercise 5: Build an MLP and prepare MNIST

Stack three linear layers with ReLU activations, then set up the MNIST loaders plus optimizer/state for pretraining.

In [7]:
class MultilayerPerceptron(nn.Module):
    def __init__(self, num_features, num_hidden_1, num_hidden_2, num_classes):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(num_features, num_hidden_1),
            nn.ReLU(),
            nn.Linear(num_hidden_1, num_hidden_2),
            nn.ReLU(),
            nn.Linear(num_hidden_2, num_classes)
        )

    def forward(self, x):
        # On aplatit l'image MNIST (28x28) en vecteur (784)
        x = x.view(-1, 28*28)
        x = self.layers(x)
        return x

In [8]:
# Architecture
num_features = 784 # 28x28 pixels
num_hidden_1 = 128
num_hidden_2 = 64
num_classes = 10

# Settings
learning_rate = 0.005
num_epochs = 2

model = MultilayerPerceptron(
    num_features=num_features,
    num_hidden_1=num_hidden_1,
    num_hidden_2=num_hidden_2,
    num_classes=num_classes,
)

model.to(DEVICE)
optimizer_pretrained = torch.optim.Adam(model.parameters(), lr=learning_rate)
print(f"Device: {DEVICE}")
print(model)
print(f"Optimizer: {optimizer_pretrained}")

Device: cpu
MultilayerPerceptron(
  (layers): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.005
    maximize: False
    weight_decay: 0
)


## Loading dataset

In [9]:
BATCH_SIZE = 64

train_dataset = datasets.MNIST(root='data', train=True, transform=transforms.ToTensor(), download=True)
test_dataset = datasets.MNIST(root='data', train=False, transform=transforms.ToTensor(), download=True)

train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)

for images, labels in train_loader:
    print('Image batch dimensions:', images.shape)
    print('Image label dimensions:', labels.shape)
    break

100%|██████████| 9.91M/9.91M [00:00<00:00, 41.6MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.13MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 10.2MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.91MB/s]


Image batch dimensions: torch.Size([64, 1, 28, 28])
Image label dimensions: torch.Size([64])


## Define evaluation

In [10]:
def compute_accuracy(model, data_loader, device):
    model.eval()
    correct_pred, num_examples = 0, 0
    with torch.no_grad():
        for features, targets in data_loader:
            features = features.to(device)
            targets = targets.to(device)
            logits = model(features)
            _, predicted_labels = torch.max(logits, 1)
            num_examples += targets.size(0)
            correct_pred += (predicted_labels == targets).sum().item()
    return (correct_pred / num_examples) * 100

## Training

In [11]:
def train(num_epochs, model, optimizer, train_loader, device):
    start_time = time.time()
    for epoch in range(num_epochs):
        model.train()
        for batch_idx, (features, targets) in enumerate(train_loader):
            features = features.to(device)
            targets = targets.to(device)

            logits = model(features)
            loss = F.cross_entropy(logits, targets)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if not batch_idx % 400:
                print('Epoch: %03d/%03d|Batch %03d/%03d| Loss: %.4f' % (epoch+1, num_epochs, batch_idx, len(train_loader), loss))

        with torch.set_grad_enabled(False):
            print('Epoch: %03d/%03d training accuracy: %.2f%%' % (epoch+1, num_epochs, compute_accuracy(model, train_loader, device)))

        print('Time elapsed: %.2f min' % ((time.time() - start_time)/60))
    print('Total Training Time: %.2f min' % ((time.time() - start_time)/60))

In [ ]:
train(num_epochs, model, optimizer_pretrained, train_loader, DEVICE)
print(f'Test accuracy: {compute_accuracy(model, test_loader, DEVICE):.2f}%')

# Replacing Linear with LoRA Layers

In [12]:
model_lora = copy.deepcopy(model)

# Application de LoRA sur toutes les couches linéaires du MLP
model_lora.layers[0] = LinearWithLoRAMerged(model_lora.layers[0], rank=4, alpha=8)
model_lora.layers[2] = LinearWithLoRAMerged(model_lora.layers[2], rank=4, alpha=8)
model_lora.layers[4] = LinearWithLoRAMerged(model_lora.layers[4], rank=4, alpha=8)

model_lora.to(DEVICE)
optimizer_lora = torch.optim.Adam(model_lora.parameters(), lr=learning_rate)

print(model_lora)
print(f'Test accuracy orig model: {compute_accuracy(model, test_loader, DEVICE):.2f}%')
print(f'Test accuracy LoRA model: {compute_accuracy(model_lora, test_loader, DEVICE):.2f}%')

MultilayerPerceptron(
  (layers): Sequential(
    (0): LinearWithLoRAMerged(
      (linear): Linear(in_features=784, out_features=128, bias=True)
      (lora): LoRALayer()
    )
    (1): ReLU()
    (2): LinearWithLoRAMerged(
      (linear): Linear(in_features=128, out_features=64, bias=True)
      (lora): LoRALayer()
    )
    (3): ReLU()
    (4): LinearWithLoRAMerged(
      (linear): Linear(in_features=64, out_features=10, bias=True)
      (lora): LoRALayer()
    )
  )
)
Test accuracy orig model: 8.91%
Test accuracy LoRA model: 8.91%


## Freezing the Original Linear Layers

In [13]:
def freeze_linear_layers(model):
    for child in model.children():
        if isinstance(child, nn.Linear):
            for param in child.parameters():
                param.requires_grad = False
        else:
            freeze_linear_layers(child)

freeze_linear_layers(model_lora)
for name, param in model_lora.named_parameters():
    print(f'{name}:{param.requires_grad}')

layers.0.linear.weight:False
layers.0.linear.bias:False
layers.0.lora.A:True
layers.0.lora.B:True
layers.2.linear.weight:False
layers.2.linear.bias:False
layers.2.lora.A:True
layers.2.lora.B:True
layers.4.linear.weight:False
layers.4.linear.bias:False
layers.4.lora.A:True
layers.4.lora.B:True


In [14]:
optimizer_lora = torch.optim.Adam(model_lora.parameters(), lr=learning_rate)
train(num_epochs, model_lora, optimizer_lora, train_loader, DEVICE)
print(f'Test accuracy LoRA finetune: {compute_accuracy(model_lora, test_loader, DEVICE):.2f}%')

print(f'Test accuracy orig model:{compute_accuracy(model, test_loader, DEVICE):.2f}%')
print(f'Test accuracy LoRA model:{compute_accuracy(model_lora, test_loader, DEVICE):.2f}%')

/tmp/ipykernel_1853/673898811.py:17: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print('Epoch: %03d/%03d|Batch %03d/%03d| Loss: %.4f' % (epoch+1, num_epochs, batch_idx, len(train_loader), loss))


Epoch: 001/002|Batch 000/938| Loss: 2.3202
Epoch: 001/002|Batch 400/938| Loss: 0.6623
Epoch: 001/002|Batch 800/938| Loss: 0.7056
Epoch: 001/002 training accuracy: 82.54%
Time elapsed: 0.36 min
Epoch: 002/002|Batch 000/938| Loss: 0.3651
Epoch: 002/002|Batch 400/938| Loss: 0.3706
Epoch: 002/002|Batch 800/938| Loss: 0.3948
Epoch: 002/002 training accuracy: 86.33%
Time elapsed: 0.67 min
Total Training Time: 0.67 min
Test accuracy LoRA finetune: 86.37%
Test accuracy orig model:8.91%
Test accuracy LoRA model:86.37%
